# Competencia: tus neuronas y capas contra la tecnología normal

Pone a competir tus piezas de investigación del cuaderno 02 (IDA, seno, SMA, HMN, RMB…) contra los modelos que se usan hoy:

| Datos | Tecnología normal (referencia) | Conjuntos (Hugging Face) |
|---|---|---|
| Imágenes | CNN con BatchNorm | MNIST, Fashion-MNIST, CIFAR-10 |
| Texto | TextCNN y Transformer pequeño | reseñas en español, AG News, IMDB |
| Series de tiempo | CNN 1D y LSTM (+ predicción ingenua) | electricidad de Texas, tipo de cambio, río Saugeen |

Cada pieza compite de dos formas:
- **en la red densa**;
- **dentro de la tecnología normal**: en la parte final y, si es una activación, también reemplazando a ReLU por dentro.

Para cada pieza, el informe da:
- una **tabla de posiciones** por tarea;
- **cuánto mejor o peor** sale frente a la referencia;
- si **mejora de verdad** su arquitectura: varias semillas, prueba estadística y corrección por comparaciones múltiples.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. Todo se guarda en `/content/` (se borra al cerrar Colab; descarga el informe al final).

## 1. Traer el repositorio y preparar todo

El repo es privado. Guarda un token de GitHub de **solo lectura** en *Secretos de Colab* (ícono de llave a la izquierda) con el nombre `GITHUB_TOKEN`. **Nunca lo pegues en una celda.** Si tienes un token de Hugging Face, guárdalo como `HF_TOKEN` (es opcional: solo acelera las descargas).

In [ ]:
import os, subprocess, torch
from google.colab import userdata

os.environ["HF_HOME"] = "/content/hf"                 # archivos descargados de Hugging Face
os.environ["LABORATORIO_CACHE"] = "/content/cache"   # datos ya preparados (la 2.ª vez cargan en segundos)
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # sin token también funciona

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO hay GPU: cambia el tipo de entorno (irá muy lento)")

## 2. Prueba rápida (≈ 15–30 min con GPU)

Una tarea por tipo (MNIST, reseñas en español, electricidad), 3 000 ejemplos y 2 semillas. Sirve para ver que todo funciona y tener una primera idea.

In [ ]:
!python -m laboratorio.banco --competencia --rapido --guardar /content/modelos --salida /content/resultados

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/content/resultados/competencia.md", encoding="utf-8").read()))

## 3. Competencia completa por tipo de dato

Con 10 000 ejemplos y 3 semillas, cada tipo puede tardar de 1 a 3 horas con GPU. Corre solo el que te interese; si Colab se desconecta, vuelve a correr la celda 1 y luego esta (los datos ya preparados se vuelven a descargar, porque `/content` se borra).

Para elegir piezas: `--pieza ida --pieza seno`. Para más semillas (resultados más firmes): `--semillas 5`.

In [ ]:
!python -m laboratorio.banco --competencia --tipo imagen --guardar /content/modelos --salida /content/resultados/imagen

In [ ]:
!python -m laboratorio.banco --competencia --tipo texto --guardar /content/modelos --salida /content/resultados/texto

In [ ]:
!python -m laboratorio.banco --competencia --tipo serie --guardar /content/modelos --salida /content/resultados/serie

In [ ]:
import glob
for ruta in sorted(glob.glob("/content/resultados/*/competencia.md")):
    display(Markdown(open(ruta, encoding="utf-8").read()))

## 4. Bajar los resultados a tu computadora

`/content` se borra al cerrar Colab: descarga el informe (y los modelos si los quieres conservar).

In [ ]:
!cd /content && zip -qr resultados.zip resultados modelos
from google.colab import files
files.download("/content/resultados.zip")

## 5. Recuperar un modelo guardado

Los modelos se guardan como `pieza__tarea__arquitectura__s<semilla>.pt`. Se cargan con `weights_only=True`: solo pesos, nunca código.

In [ ]:
from laboratorio.banco.evaluar import cargar_modelo, _predecir, _medir
from laboratorio.banco.tareas import cargar_tarea

modelo = cargar_modelo("/content/modelos/seno__mnist__cnn__s0.pt")
tarea = cargar_tarea("mnist")
print("exactitud en prueba:", _medir(tarea, _predecir(modelo, tarea.x_prueba), tarea.y_prueba))

## 6. Banco rápido sin GPU (tareas pequeñas)

Las 9 piezas contra la red densa estándar en 6 tareas pequeñas (≈ 5 min en CPU):

In [ ]:
!python -m laboratorio.banco --todo --semillas 5 --salida /content/resultados/pequenas

## 7. Agregar una idea nueva

1. Escribe tu capa como `nn.Module(entradas, salidas)` y agrégala a `PIEZAS` en `laboratorio/banco/piezas.py`.
2. Si es una activación, dale también `activacion=lambda canales: TuActivacion(canales)` para que entre dentro de las CNN.
3. Córrela: `!python -m laboratorio.banco --competencia --rapido --pieza tu_pieza`.